# Sistema Integral de Gestión de Clientes, Servicios y Reservas
**Software FJ** — UNAD, Programación 213023, Fase 4 (Anexo 3, Ejercicio 1)

**Resultado de aprendizaje:** implementar el manejo de excepciones en aplicaciones orientadas a objetos buscando estabilidad y robustez, con gestión adecuada de errores.

**Contenido del cuaderno (ejecutar en orden, de arriba hacia abajo):**
1. Configuración del sistema de logs
2. Jerarquía de excepciones personalizadas
3. Clase abstracta `Entidad` y clase `Cliente`
4. Clase abstracta `Servicio` y sus tres especializaciones
5. Clase `Reserva` con confirmación, cancelación y procesamiento
6. Clase `GestorSistema` con listas internas y control de duplicados
7. Pruebas: 16 operaciones válidas e inválidas con registro en el log

In [1]:
import logging                          # Importa módulo estándar de logs
from abc import ABC, abstractmethod     # Importa clases abstractas estándar

RUTA_LOG = "sistema_fj.log"               # Archivo único de eventos y errores

try:                                      # Intenta configurar logger en archivo
    logging.basicConfig(                  # Configura formato, nivel y destino
        filename=RUTA_LOG,                # Nombre del archivo de registro
        filemode="w",                     # Limpia el log en cada ejecución
        level=logging.INFO,               # Registra desde nivel INFO hacia arriba
        format="%(asctime)s | %(levelname)s | %(message)s",  # Formato línea
        force=True,                       # Evita handlers duplicados al reejecutar
    )
    logger = logging.getLogger("SoftwareFJ")   # Logger nombrado del proyecto
    logger.info("Sistema Software FJ iniciado")    # Registra evento de inicio
except OSError as error:                  # Captura fallos al crear el archivo
    logger = logging.getLogger("SoftwareFJ")       # Logger de respaldo
    logger.addHandler(logging.StreamHandler())     # Registra por consola
    logger.error("Fallo de archivo de log: %s", error)  # Informa el fallo

## 1. Excepciones personalizadas
Cinco excepciones propias que heredan de una base del proyecto, para distinguir errores de validación, servicios no disponibles, operaciones no permitidas, cálculos inconsistentes y registros duplicados.

In [2]:
class ExcepcionProyecto(Exception):                # Excepción base personalizada
    pass                                             # Reutiliza mensaje de Exception


class ExcepcionValidacionDatos(ExcepcionProyecto):     # Error de datos inválidos
    pass                                               # Hereda comportamiento base


class ExcepcionServicioNoDisponible(ExcepcionProyecto):  # Servicio inhabilitado
    pass                                                 # Hereda comportamiento base


class ExcepcionOperacionNoPermitida(ExcepcionProyecto):  # Acción no permitida
    pass                                                 # Hereda comportamiento base


class ExcepcionCalculoInconsistente(ExcepcionProyecto):  # Resultado inválido
    pass                                                 # Hereda comportamiento base


class ExcepcionRegistroDuplicado(ExcepcionProyecto):   # Registro ya existente
    pass                                                 # Hereda comportamiento base

## 2. Clase abstracta `Entidad`
Define el identificador único y obliga a cada subclase a implementar `resumen()`. El identificador se asigna solo después de validar los datos, para que los objetos inválidos no consuman números.

In [3]:
class Entidad(ABC):                           # Clase abstracta general del sistema
    _contador_ids = 0                         # Contador compartido de identificadores

    def __init__(self):                       # Constructor base de entidades
        self._id = None                       # Identificador aún sin asignar

    def _asignar_id(self):                    # Asigna identificador tras validar
        Entidad._contador_ids += 1            # Incrementa contador de instancias
        self._id = Entidad._contador_ids      # Asigna identificador único interno

    @property                                 # Expone el método como propiedad
    def id(self):                             # Expone identificador solo lectura
        return self._id                       # Devuelve identificador interno

    @abstractmethod                           # Obliga implementarlo en subclases
    def resumen(self):                        # Obliga resumen en subclases
        pass                                  # Sin implementación en la base

## 3. Clase `Cliente`
Encapsulamiento mediante propiedades con setters validados: tipos de datos, valores vacíos, rangos de longitud, dígitos y formato de correo validado con expresión regular (`re`).

In [4]:
import re                                                 # Importa expresiones regulares


class Cliente(Entidad):                                    # Entidad con encapsulamiento
    PATRON_CORREO = re.compile(r"^[\w.+-]+@[\w-]+(\.[\w-]+)+$")  # Formato correo

    def __init__(self, cedula, nombre, correo, telefono):    # Recibe datos personales
        super().__init__()                                   # Inicializa identificador base
        self._cedula = None                                  # Atributo interno de cédula
        self._nombre = None                                  # Atributo interno de nombre
        self._correo = None                                  # Atributo interno de correo
        self._telefono = None                                # Atributo interno de teléfono
        self.cedula = cedula                                 # Valida mediante propiedad
        self.nombre = nombre                                 # Valida mediante propiedad
        self.correo = correo                                 # Valida mediante propiedad
        self.telefono = telefono                             # Valida mediante propiedad
        self._asignar_id()                                   # Asigna ID tras validar todo

    @staticmethod                                            # Método de clase auxiliar
    def _validar_texto(valor, campo, minimo=2, maximo=80):   # Valida cadenas de texto
        if not isinstance(valor, str):                       # Verifica tipo cadena
            raise ExcepcionValidacionDatos(f"{campo} debe ser texto")  # Error de tipo
        limpio = valor.strip()                               # Elimina espacios extremos
        if not limpio:                                       # Detecta valor vacío
            raise ExcepcionValidacionDatos(f"{campo} no puede estar vacío")  # Error vacío
        if not (minimo <= len(limpio) <= maximo):            # Verifica rango longitud
            raise ExcepcionValidacionDatos(f"{campo} fuera de rango")  # Error de rango
        return limpio                                        # Devuelve texto normalizado

    @property                                                # Expone cédula lectura
    def cedula(self):                                        # Getter de cédula
        return self._cedula                                  # Devuelve cédula interna

    @cedula.setter                                           # Setter validado cédula
    def cedula(self, valor):                                 # Setter validado de cédula
        texto = self._validar_texto(valor, "Cédula", 5, 20)  # Valida texto y longitud
        if not texto.isdigit():                              # Exige solo dígitos
            raise ExcepcionValidacionDatos("Cédula solo puede tener dígitos")  # Error
        self._cedula = texto                                 # Guarda cédula validada

    @property                                                # Expone nombre lectura
    def nombre(self):                                        # Getter de nombre
        return self._nombre                                  # Devuelve nombre interno

    @nombre.setter                                           # Setter validado nombre
    def nombre(self, valor):                                 # Setter validado de nombre
        self._nombre = self._validar_texto(valor, "Nombre")  # Valida y guarda nombre

    @property                                                # Expone correo lectura
    def correo(self):                                        # Getter de correo
        return self._correo                                  # Devuelve correo interno

    @correo.setter                                           # Setter validado correo
    def correo(self, valor):                                 # Setter validado de correo
        texto = self._validar_texto(valor, "Correo", 5, 120)  # Valida texto y longitud
        if not self.PATRON_CORREO.match(texto):              # Verifica formato con regex
            raise ExcepcionValidacionDatos("Correo con formato inválido")  # Error formato
        self._correo = texto                                 # Guarda correo validado

    @property                                                # Expone teléfono lectura
    def telefono(self):                                      # Getter de teléfono
        return self._telefono                                # Devuelve teléfono interno

    @telefono.setter                                         # Setter validado teléfono
    def telefono(self, valor):                               # Setter validado de teléfono
        texto = self._validar_texto(valor, "Teléfono", 7, 15)  # Valida texto y longitud
        if not texto.isdigit():                              # Exige solo dígitos
            raise ExcepcionValidacionDatos("Teléfono solo puede tener dígitos")  # Error
        self._telefono = texto                               # Guarda teléfono validado

    def resumen(self):                                       # Implementa método abstracto
        return f"Cliente #{self.id}: {self.nombre} ({self.cedula})"  # Ficha corta

## 4. Clase abstracta `Servicio`
Define el precio base, la disponibilidad (con validación estricta de booleano) y los validadores reutilizables. Declara los métodos abstractos `calcular_costo()` y `descripcion()` que cada servicio sobrescribe (polimorfismo).

In [5]:
class Servicio(Entidad, ABC):                                 # Clase abstracta de servicios
    IMPUESTO_BASE = 0.19                                      # Impuesto por defecto (19%)

    def __init__(self, codigo, nombre, precio_base, disponible=True):  # Constructor
        super().__init__()                                    # Inicializa identificador base
        self._codigo = self._validar_texto_simple(codigo, "Código")      # Valida código
        self._nombre = self._validar_texto_simple(nombre, "Nombre servicio")  # Valida
        self.precio_base = precio_base                        # Valida mediante propiedad
        self.disponible = disponible                          # Valida mediante propiedad

    @property                                                 # Expone código solo lectura
    def codigo(self):                                         # Getter de código
        return self._codigo                                   # Devuelve código interno

    @staticmethod                                             # Método de clase auxiliar
    def _validar_texto_simple(valor, campo):                  # Valida textos cortos
        if not isinstance(valor, str) or not valor.strip():   # Rechaza texto vacío
            raise ExcepcionValidacionDatos(f"{campo} inválido")  # Error de validación
        return valor.strip()                                  # Devuelve texto normalizado

    @staticmethod                                             # Método de clase auxiliar
    def _validar_entero(valor, campo, minimo):                # Valida enteros con mínimo
        try:                                                  # Intenta conversión entera
            numero = int(valor)                               # Conversión explícita
        except (TypeError, ValueError) as error:              # Captura tipos inválidos
            raise ExcepcionValidacionDatos(f"{campo} debe ser entero") from error  # Encadena
        if numero < minimo:                                   # Verifica mínimo permitido
            raise ExcepcionValidacionDatos(f"{campo} debe ser >= {minimo}")  # Error
        return numero                                         # Devuelve entero validado

    @property                                                 # Expone precio solo lectura
    def precio_base(self):                                    # Getter de precio base
        return self._precio_base                              # Devuelve precio interno

    @precio_base.setter                                       # Setter validado precio
    def precio_base(self, valor):                             # Setter validado de precio
        try:                                                  # Intenta conversión numérica
            numero = float(valor)                             # Conversión explícita
        except (TypeError, ValueError) as error:              # Captura tipos inválidos
            raise ExcepcionValidacionDatos("Precio debe ser numérico") from error  # Encadena
        if numero <= 0:                                       # Rechaza precio no positivo
            raise ExcepcionValidacionDatos("Precio debe ser mayor que cero")  # Error
        self._precio_base = numero                            # Guarda precio validado

    @property                                                 # Expone bandera solo lectura
    def disponible(self):                                     # Getter de disponibilidad
        return self._disponible                               # Devuelve bandera interna

    @disponible.setter                                        # Setter validado bandera
    def disponible(self, valor):                              # Setter de disponibilidad
        if not isinstance(valor, bool):                       # Exige booleano real
            raise ExcepcionValidacionDatos("Disponible debe ser booleano")  # Error tipo
        self._disponible = valor                              # Guarda bandera validada

    def _validar_factor(self, valor, campo, maximo):          # Valida factores 0..máximo
        try:                                                  # Intenta conversión numérica
            numero = float(valor)                             # Conversión explícita
        except (TypeError, ValueError) as error:              # Captura tipos inválidos
            raise ExcepcionCalculoInconsistente(f"{campo} numérico") from error  # Encadena
        if not (0 <= numero <= maximo):                       # Verifica rango permitido
            raise ExcepcionCalculoInconsistente(f"{campo} fuera de rango")  # Error
        return numero                                         # Devuelve factor validado

    def validar_parametros(self):                             # Valida coherencia interna
        if self.precio_base <= 0:                             # Precio debe ser positivo
            raise ExcepcionCalculoInconsistente("Precio base inconsistente")  # Error

    @abstractmethod                                           # Obliga sobrescribirlo
    def calcular_costo(self, descuento=0.0, impuesto=IMPUESTO_BASE):  # Costo total
        pass                                                  # Cada subclase lo sobrescribe

    @abstractmethod                                           # Obliga sobrescribirlo
    def descripcion(self):                                    # Descripción del servicio
        pass                                                  # Cada subclase lo sobrescribe

    def resumen(self):                                        # Implementa método abstracto
        estado = "disponible" if self.disponible else "no disponible"  # Texto de estado
        return f"{self._codigo} | {self._nombre} | {estado}"  # Línea resumen

## 5. Servicios especializados (herencia y polimorfismo)
`ReservaSala`, `AlquilerEquipo` y `AsesoriaEspecializada` sobrescriben `calcular_costo()` con distinta lógica y parámetros opcionales (sobrecarga simulada en Python). Todos reciben la duración en **horas** como parámetro, sin duplicarla dentro de la reserva.

In [6]:
class ReservaSala(Servicio):                                  # Servicio de reserva de salas
    def __init__(self, codigo, nombre, precio_base, sala, capacidad):  # Constructor
        super().__init__(codigo, nombre, precio_base)         # Inicializa servicio base
        self._sala = self._validar_texto_simple(sala, "Sala")  # Valida nombre de sala
        self._capacidad = self._validar_entero(capacidad, "Capacidad", 1)  # Valida
        self._asignar_id()                                    # Asigna ID tras validar todo

    def calcular_costo(self, descuento=0.0, impuesto=Servicio.IMPUESTO_BASE,  # Variante
                       horas=1):                              # Duración en horas
        descuento = self._validar_factor(descuento, "Descuento", 1.0)  # Valida rango
        impuesto = self._validar_factor(impuesto, "Impuesto", 1.0)  # Valida rango
        horas = self._validar_entero(horas, "Horas", 1)       # Valida horas reservadas
        subtotal = self.precio_base * horas                   # Costo por horas reservadas
        return subtotal * (1 - descuento) * (1 + impuesto)    # Aplica descuento e impuesto

    def descripcion(self):                                    # Describe sala y capacidad
        return (f"Sala {self._sala} con capacidad para {self._capacidad} "
                f"personas")                                  # Texto descriptivo

In [7]:
class AlquilerEquipo(Servicio):                               # Servicio de alquiler de equipos
    def __init__(self, codigo, nombre, precio_base, equipo, cantidad):  # Constructor
        super().__init__(codigo, nombre, precio_base)         # Inicializa servicio base
        self._equipo = self._validar_texto_simple(equipo, "Equipo")  # Valida equipo
        self._cantidad = self._validar_entero(cantidad, "Cantidad", 1)  # Valida unidades
        self._asignar_id()                                    # Asigna ID tras validar todo

    def calcular_costo(self, descuento=0.0, impuesto=Servicio.IMPUESTO_BASE,  # Variante
                       horas=1):                              # Duración en horas
        descuento = self._validar_factor(descuento, "Descuento", 1.0)  # Valida rango
        impuesto = self._validar_factor(impuesto, "Impuesto", 1.0)  # Valida rango
        horas = self._validar_entero(horas, "Horas", 1)       # Valida horas de alquiler
        subtotal = self.precio_base * horas * self._cantidad  # Costo por horas y unidades
        return subtotal * (1 - descuento) * (1 + impuesto)    # Total final del alquiler

    def descripcion(self):                                    # Describe equipo y unidades
        return (f"Alquiler de {self._equipo}, {self._cantidad} "
                f"unidad(es)")                                # Texto descriptivo

In [8]:
class AsesoriaEspecializada(Servicio):                        # Servicio de asesorías
    RECARGO_SENIOR = 0.25                                     # Recargo por experto senior

    def __init__(self, codigo, nombre, precio_base, tema, experto, senior=False):  # Constructor
        super().__init__(codigo, nombre, precio_base)         # Inicializa servicio base
        self._tema = self._validar_texto_simple(tema, "Tema")  # Valida tema de asesoría
        self._experto = self._validar_texto_simple(experto, "Experto")  # Valida experto
        self._senior = bool(senior)                           # Marca nivel del experto
        self._asignar_id()                                    # Asigna ID tras validar todo

    def calcular_costo(self, descuento=0.0, impuesto=Servicio.IMPUESTO_BASE,  # Variante
                       horas=1):                              # Duración en horas
        descuento = self._validar_factor(descuento, "Descuento", 1.0)  # Valida rango
        impuesto = self._validar_factor(impuesto, "Impuesto", 1.0)  # Valida rango
        horas = self._validar_entero(horas, "Horas", 1)       # Valida horas de asesoría
        subtotal = self.precio_base * horas                   # Costo por horas asesoría
        if self._senior:                                      # Evalúa recargo por senior
            subtotal *= 1 + self.RECARGO_SENIOR               # Incrementa subtotal 25%
        return subtotal * (1 - descuento) * (1 + impuesto)    # Total final de asesoría

    def descripcion(self):                                    # Describe tema y experto
        nivel = "senior" if self._senior else "estándar"      # Texto de nivel experto
        return f"Asesoría en {self._tema} con experto {nivel} {self._experto}"  # Texto

## 6. Clase `Reserva`
Integra cliente, servicio, duración (en horas) y estado. Implementa `confirmar()`, `cancelar()` y `procesar()` con bloques try/except/else/finally y registro de errores en el log. En `procesar()` el costo se calcula con la duración de la reserva.

In [9]:
class Reserva(Entidad):                                       # Integra cliente y servicio
    ESTADOS = ("PENDIENTE", "CONFIRMADA", "CANCELADA", "COMPLETADA")  # Flujo permitido

    def __init__(self, cliente, servicio, duracion, estado="PENDIENTE"):  # Constructor
        super().__init__()                                    # Inicializa identificador base
        self._cliente = None                                  # Atributo interno de cliente
        self._servicio = None                                 # Atributo interno de servicio
        self._duracion = 0                                    # Atributo interno duración
        self._estado = ""                                     # Atributo interno de estado
        self.cliente = cliente                                # Valida mediante propiedad
        self.servicio = servicio                              # Valida mediante propiedad
        self.duracion = duracion                              # Valida mediante propiedad
        self.estado = estado                                  # Valida mediante propiedad
        self._asignar_id()                                    # Asigna ID tras validar todo

    @property                                                 # Expone cliente solo lectura
    def cliente(self):                                        # Getter de cliente
        return self._cliente                                  # Devuelve cliente interno

    @cliente.setter                                           # Setter validado cliente
    def cliente(self, valor):                                 # Setter validado de cliente
        if not isinstance(valor, Cliente):                    # Exige instancia de Cliente
            raise ExcepcionValidacionDatos("Se requiere un cliente válido")  # Error
        self._cliente = valor                                 # Guarda cliente validado

    @property                                                 # Expone servicio solo lectura
    def servicio(self):                                       # Getter de servicio
        return self._servicio                                 # Devuelve servicio interno

    @servicio.setter                                          # Setter validado servicio
    def servicio(self, valor):                                # Setter validado de servicio
        if not isinstance(valor, Servicio):                   # Exige instancia de Servicio
            raise ExcepcionValidacionDatos("Se requiere un servicio válido")  # Error
        self._servicio = valor                                # Guarda servicio validado

    @property                                                 # Expone duración solo lectura
    def duracion(self):                                       # Getter de duración
        return self._duracion                                 # Devuelve duración interna

    @duracion.setter                                          # Setter validado duración
    def duracion(self, valor):                                # Setter validado de duración
        self._duracion = Servicio._validar_entero(valor, "Duración", 1)  # Valida y guarda

    @property                                                 # Expone estado solo lectura
    def estado(self):                                         # Getter de estado
        return self._estado                                   # Devuelve estado interno

    @estado.setter                                            # Setter validado estado
    def estado(self, valor):                                  # Setter validado de estado
        if valor not in self.ESTADOS:                         # Rechaza estados desconocidos
            raise ExcepcionOperacionNoPermitida(f"Estado inválido: {valor}")  # Error
        self._estado = valor                                  # Guarda estado validado

    def confirmar(self):                                      # Confirma reserva pendiente
        if self._estado != "PENDIENTE":                       # Solo pendientes se confirman
            raise ExcepcionOperacionNoPermitida("Solo se confirman reservas pendientes")
        if not self._servicio.disponible:                     # Rechaza servicio inhabilitado
            raise ExcepcionServicioNoDisponible("El servicio no está disponible")  # Error
        self._estado = "CONFIRMADA"                           # Cambia estado a confirmada
        return f"Reserva #{self.id} confirmada para {self._cliente.nombre}"  # Resultado

    def cancelar(self):                                       # Cancela reserva activa
        if self._estado not in ("PENDIENTE", "CONFIRMADA"):   # Estados que admiten cancelar
            raise ExcepcionOperacionNoPermitida("La reserva no se puede cancelar")  # Error
        self._estado = "CANCELADA"                            # Cambia estado a cancelada
        return f"Reserva #{self.id} cancelada correctamente"  # Mensaje de confirmación

    def procesar(self):                                       # Procesa cobro y completa
        try:                                                  # Intento de procesamiento
            if self._estado != "CONFIRMADA":                  # Requiere reserva confirmada
                raise ExcepcionOperacionNoPermitida("Se requiere reserva confirmada")
            total = self._servicio.calcular_costo(horas=self._duracion)  # Costo con duración
        except ExcepcionProyecto as error:                    # Captura errores del dominio
            logger.error("Fallo al procesar reserva #%s: %s", self.id, error)  # Registra
            raise                                             # Propaga para tratamiento externo
        else:                                                 # Solo si no hubo errores
            self._estado = "COMPLETADA"                       # Marca reserva como completada
            return f"Reserva #{self.id} procesada por ${total:,.2f}"  # Resultado exitoso
        finally:                                              # Siempre se ejecuta
            logger.info("Intento de procesar reserva #%s", self.id)  # Registra intento

    def resumen(self):                                        # Implementa método abstracto
        return (f"Reserva #{self.id} | {self._cliente.nombre} | "
                f"{self._servicio.resumen()} | {self._estado}")  # Línea resumen

## 7. Clase `GestorSistema`
Centraliza el sistema con listas internas de clientes, servicios y reservas. Los métodos `registrar_cliente`, `registrar_servicio` y `registrar_reserva` validan duplicados (por cédula y por código) y lanzan `ExcepcionRegistroDuplicado`.

In [10]:
class GestorSistema:                                  # Gestor central con listas internas
    def __init__(self):                               # Constructor del gestor
        self._clientes = []                           # Lista interna de clientes
        self._servicios = []                          # Lista interna de servicios
        self._reservas = []                           # Lista interna de reservas

    @property                                         # Expone clientes solo lectura
    def clientes(self):                               # Getter de clientes registrados
        return tuple(self._clientes)                  # Devuelve copia inmutable

    @property                                         # Expone servicios solo lectura
    def servicios(self):                              # Getter de servicios registrados
        return tuple(self._servicios)                 # Devuelve copia inmutable

    @property                                         # Expone reservas solo lectura
    def reservas(self):                               # Getter de reservas registradas
        return tuple(self._reservas)                  # Devuelve copia inmutable

    def registrar_cliente(self, cliente):             # Registra cliente sin duplicados
        if not isinstance(cliente, Cliente):          # Exige instancia de Cliente
            raise ExcepcionValidacionDatos("Se requiere un cliente válido")  # Error tipo
        duplicada = any(c.cedula == cliente.cedula    # Busca cédula ya registrada
                        for c in self._clientes)      # Recorre lista interna
        if duplicada:                                 # Detectó cédula repetida
            raise ExcepcionRegistroDuplicado(f"Cédula ya registrada: {cliente.cedula}")  # Error
        self._clientes.append(cliente)                # Agrega cliente a lista interna
        return cliente                                # Devuelve cliente registrado

    def registrar_servicio(self, servicio):           # Registra servicio sin duplicados
        if not isinstance(servicio, Servicio):        # Exige instancia de Servicio
            raise ExcepcionValidacionDatos("Se requiere un servicio válido")  # Error tipo
        duplicado = any(s.codigo == servicio.codigo   # Busca código ya registrado
                        for s in self._servicios)     # Recorre lista interna
        if duplicado:                                 # Detectó código repetido
            raise ExcepcionRegistroDuplicado(f"Código ya registrado: {servicio.codigo}")  # Error
        self._servicios.append(servicio)              # Agrega servicio a lista interna
        return servicio                               # Devuelve servicio registrado

    def registrar_reserva(self, reserva):             # Registra reserva sin duplicados
        if not isinstance(reserva, Reserva):          # Exige instancia de Reserva
            raise ExcepcionValidacionDatos("Se requiere una reserva válida")  # Error tipo
        for existente in self._reservas:              # Recorre reservas registradas
            mismo_cliente = existente.cliente is reserva.cliente  # Compara clientes
            mismo_servicio = existente.servicio is reserva.servicio  # Compara servicios
            activa = existente.estado in ("PENDIENTE", "CONFIRMADA")  # Verifica activa
            if mismo_cliente and mismo_servicio and activa:  # Duplicado activo detectado
                raise ExcepcionRegistroDuplicado("Reserva activa duplicada")  # Error duplicado
        self._reservas.append(reserva)                # Agrega reserva a lista interna
        return reserva                                # Devuelve reserva registrada

## 8. Pruebas del sistema (16 operaciones)
Se ejecutan operaciones válidas e inválidas a propósito usando el `GestorSistema`. Cada error se captura, se registra en `sistema_fj.log` y el sistema continúa activo. La última celda muestra el contenido del log.

In [11]:
def ejecutar_operacion(numero, titulo, operacion):     # Ejecuta y reporta operación
    print(f"\n--- Operación {numero}: {titulo} ---")   # Encabezado visible
    try:                                                 # Intento de operación
        resultado = operacion()                          # Ejecuta operación recibida
    except ExcepcionProyecto as error:                   # Errores propios del dominio
        logger.error("Operación %s falló: %s", numero, error)  # Registra error en log
        print(f"[ERROR CONTROLADO] {type(error).__name__}: {error}")  # Mensaje claro
    except (TypeError, ValueError) as error:             # Errores de tipos de datos
        logger.error("Operación %s falló: %s", numero, error)  # Registra error en log
        print(f"[ERROR CONTROLADO] {error}")             # Mensaje claro al usuario
    except OSError as error:                             # Errores de archivos del sistema
        logger.error("Operación %s falló: %s", numero, error)  # Registra error en log
        print(f"[ERROR DE ARCHIVO CONTROLADO] {error}")  # Mensaje claro al usuario
    except Exception as error:                           # Cualquier error inesperado
        logger.exception("Operación %s falló inesperadamente: %s", numero, error)  # Traza
        print(f"[ERROR INESPERADO CONTROLADO] {error}")  # Mensaje claro al usuario
    else:                                                # Solo si no hubo excepción
        print(f"[OK] {resultado}")                       # Muestra resultado exitoso
    finally:                                             # Siempre se ejecuta
        print("Sistema sigue activo y estable.")         # Confirma robustez del sistema

In [12]:
gestor = GestorSistema()                               # Crea gestor central del sistema
cliente_valido = None                                  # Almacena el cliente válido
servicio_sala = None                                   # Almacena el servicio de sala
asesoria = None                                        # Almacena la asesoría creada
reserva_principal = None                               # Almacena la reserva principal

def operacion_01():                                    # Registra un cliente correcto
    global cliente_valido                                # Permite guardar la instancia
    cliente = Cliente("1020304050", "Fabián Jiménez",    # Datos personales completos
                      "fabian@softwarefj.com", "3001234567")  # Correo y teléfono
    cliente_valido = gestor.registrar_cliente(cliente)   # Registra cliente en el gestor
    return cliente_valido.resumen()                      # Devuelve ficha del cliente

ejecutar_operacion(1, "Registrar cliente válido", operacion_01)  # Ejecuta operación 1


def operacion_02():                                    # Intenta cliente con nombre vacío
    cliente = Cliente("2030405060", "   ",               # Nombre inválido (solo espacios)
                      "ana@softwarefj.com", "3011111111")  # Resto de datos correctos
    return cliente.resumen()                             # Nunca se alcanza

ejecutar_operacion(2, "Cliente con nombre vacío (inválido)", operacion_02)  # Ejecuta op 2


def operacion_03():                                    # Intenta cliente con cédula inválida
    cliente = Cliente("10A20B", "Ana Ruiz",              # Cédula con letras (inválida)
                      "ana@softwarefj.com", "3011111111")  # Resto de datos correctos
    return cliente.resumen()                             # Nunca se alcanza

ejecutar_operacion(3, "Cliente con cédula inválida (inválido)", operacion_03)  # Ejecuta op 3


def operacion_04():                                    # Crea sala de reunión válida
    global servicio_sala                                 # Permite guardar la instancia
    sala = ReservaSala("SAL-01", "Sala Ejecutiva", 50000,  # Código, nombre y precio
                       "Aurora", 12)                     # Sala y capacidad
    servicio_sala = gestor.registrar_servicio(sala)      # Registra servicio en gestor
    return f"{servicio_sala.resumen()} | {servicio_sala.descripcion()}"  # Ficha completa

ejecutar_operacion(4, "Crear servicio ReservaSala válido", operacion_04)  # Ejecuta op 4


def operacion_05():                                    # Intenta equipo con precio negativo
    equipo = AlquilerEquipo("EQP-01", "Proyector 4K", -150000,  # Precio inválido (negativo)
                            "Proyector", 1)              # Equipo y cantidad válidos
    return equipo.descripcion()                          # Nunca se alcanza

ejecutar_operacion(5, "Equipo con precio negativo (inválido)", operacion_05)  # Ejecuta op 5


def operacion_06():                                    # Crea asesoría especializada válida
    global asesoria                                      # Permite guardar la instancia
    nueva = AsesoriaEspecializada("ASE-01", "Arquitectura de software", 200000,  # Datos
                                  "Microservicios", "Dra. Luna", True)  # Tema y senior
    asesoria = gestor.registrar_servicio(nueva)          # Registra servicio en gestor
    return f"{asesoria.resumen()} | {asesoria.descripcion()}"  # Ficha completa

ejecutar_operacion(6, "Crear asesoría especializada válida", operacion_06)  # Ejecuta op 6


def operacion_07():                                    # Desactiva la asesoría creada
    asesoria.disponible = False                          # Marca servicio como no disponible
    return asesoria.resumen()                            # Confirma el nuevo estado

ejecutar_operacion(7, "Marcar asesoría como no disponible", operacion_07)  # Ejecuta op 7


def operacion_08():                                    # Confirma reserva con servicio caído
    reserva = gestor.registrar_reserva(                  # Registra reserva en gestor
        Reserva(cliente_valido, asesoria, 2))            # Reserva de 2 horas
    return reserva.confirmar()                           # Falla por servicio no disponible

ejecutar_operacion(8, "Confirmar reserva de servicio no disponible (inválido)",  # Título
                   operacion_08)                         # Ejecuta operación 8


def operacion_09():                                    # Cotiza con variantes de costo
    base = servicio_sala.calcular_costo(horas=4)         # Variante con 4 horas
    con_descuento = servicio_sala.calcular_costo(horas=4, descuento=0.10)  # Descuento 10%
    completa = servicio_sala.calcular_costo(horas=4, descuento=0.10,  # Descuento
                                            impuesto=0.05)  # Impuesto 5%
    asesoria_horas = asesoria.calcular_costo(horas=3)    # Variante con horas opcionales
    return (f"Sala base ${base:,.2f} | desc ${con_descuento:,.2f} | "
            f"full ${completa:,.2f} | asesoría 3h ${asesoria_horas:,.2f}")  # Resumen

ejecutar_operacion(9, "Cotizar costos con variantes (sobrecarga)", operacion_09)  # Op 9


--- Operación 1: Registrar cliente válido ---
[OK] Cliente #1: Fabián Jiménez (1020304050)
Sistema sigue activo y estable.

--- Operación 2: Cliente con nombre vacío (inválido) ---
[ERROR CONTROLADO] ExcepcionValidacionDatos: Nombre no puede estar vacío
Sistema sigue activo y estable.

--- Operación 3: Cliente con cédula inválida (inválido) ---
[ERROR CONTROLADO] ExcepcionValidacionDatos: Cédula solo puede tener dígitos
Sistema sigue activo y estable.

--- Operación 4: Crear servicio ReservaSala válido ---
[OK] SAL-01 | Sala Ejecutiva | disponible | Sala Aurora con capacidad para 12 personas
Sistema sigue activo y estable.

--- Operación 5: Equipo con precio negativo (inválido) ---
[ERROR CONTROLADO] ExcepcionValidacionDatos: Precio debe ser mayor que cero
Sistema sigue activo y estable.

--- Operación 6: Crear asesoría especializada válida ---
[OK] ASE-01 | Arquitectura de software | disponible | Asesoría en Microservicios con experto senior Dra. Luna
Sistema sigue activo y estable.


In [13]:
def operacion_10():                                    # Crea y confirma reserva válida
    global reserva_principal                             # Permite guardar la instancia
    reserva = Reserva(cliente_valido, servicio_sala, 4)  # Reserva de 4 horas
    reserva_principal = gestor.registrar_reserva(reserva)  # Registra reserva en gestor
    return reserva_principal.confirmar()                 # Confirma la reserva creada

ejecutar_operacion(10, "Crear y confirmar reserva válida", operacion_10)  # Ejecuta op 10


def operacion_11():                                    # Intenta reconfirmar la reserva
    return reserva_principal.confirmar()                 # Falla por estado no pendiente

ejecutar_operacion(11, "Reconfirmar reserva ya confirmada (inválido)", operacion_11)  # Op 11


def operacion_12():                                    # Reserva con duración textual
    reserva = Reserva(cliente_valido, servicio_sala, "tres")  # Valor no numérico
    return reserva.resumen()                             # Nunca se alcanza

ejecutar_operacion(12, "Reserva con duración no numérica (inválido)", operacion_12)  # Op 12


def operacion_13():                                    # Calcula costo con descuento malo
    total = servicio_sala.calcular_costo(descuento=1.5)  # Descuento fuera de rango
    return f"Total: ${total:,.2f}"                       # Nunca se alcanza

ejecutar_operacion(13, "Descuento fuera de rango (inválido)", operacion_13)  # Op 13


def operacion_14():                                    # Procesa reserva confirmada
    return reserva_principal.procesar()                  # Completa cobro con duración

ejecutar_operacion(14, "Procesar reserva confirmada", operacion_14)  # Ejecuta op 14


def operacion_15():                                    # Intenta cancelar reserva terminada
    return reserva_principal.cancelar()                  # Falla por estado completada

ejecutar_operacion(15, "Cancelar reserva ya completada (inválido)", operacion_15)  # Op 15


def operacion_16():                                    # Intenta leer archivo inexistente
    with open("archivo_que_no_existe.txt", "r", encoding="utf-8") as archivo:  # Abre
        return archivo.read()                            # Nunca se alcanza

ejecutar_operacion(16, "Abrir archivo inexistente (inválido)", operacion_16)  # Op 16


--- Operación 10: Crear y confirmar reserva válida ---
[OK] Reserva #5 confirmada para Fabián Jiménez
Sistema sigue activo y estable.

--- Operación 11: Reconfirmar reserva ya confirmada (inválido) ---
[ERROR CONTROLADO] ExcepcionOperacionNoPermitida: Solo se confirman reservas pendientes
Sistema sigue activo y estable.

--- Operación 12: Reserva con duración no numérica (inválido) ---
[ERROR CONTROLADO] ExcepcionValidacionDatos: Duración debe ser entero
Sistema sigue activo y estable.

--- Operación 13: Descuento fuera de rango (inválido) ---
[ERROR CONTROLADO] ExcepcionCalculoInconsistente: Descuento fuera de rango
Sistema sigue activo y estable.

--- Operación 14: Procesar reserva confirmada ---
[OK] Reserva #5 procesada por $238,000.00
Sistema sigue activo y estable.

--- Operación 15: Cancelar reserva ya completada (inválido) ---
[ERROR CONTROLADO] ExcepcionOperacionNoPermitida: La reserva no se puede cancelar
Sistema sigue activo y estable.

--- Operación 16: Abrir archivo inexi

In [14]:
try:                                                   # Intenta abrir archivo de log
    with open(RUTA_LOG, "r", encoding="utf-8") as archivo:  # Abre el log en modo lectura
        print(archivo.read())                            # Muestra registro completo
except OSError as error:                                 # Captura errores de archivo
    print(f"No se pudo leer el log: {error}")            # Mensaje claro al usuario
finally:                                                 # Siempre se ejecuta
    print("\nFin de la demostración: el sistema nunca se detuvo.")  # Cierre de actividad

2026-11-24 04:48:03,372 | INFO | Sistema Software FJ iniciado
2026-11-24 04:48:03,429 | ERROR | Operación 2 falló: Nombre no puede estar vacío
2026-11-24 04:48:03,429 | ERROR | Operación 3 falló: Cédula solo puede tener dígitos
2026-11-24 04:48:03,429 | ERROR | Operación 5 falló: Precio debe ser mayor que cero
2026-11-24 04:48:03,429 | ERROR | Operación 8 falló: El servicio no está disponible
2026-11-24 04:48:03,434 | ERROR | Operación 11 falló: Solo se confirman reservas pendientes
2026-11-24 04:48:03,434 | ERROR | Operación 12 falló: Duración debe ser entero
2026-11-24 04:48:03,434 | ERROR | Operación 13 falló: Descuento fuera de rango
2026-11-24 04:48:03,434 | INFO | Intento de procesar reserva #5
2026-11-24 04:48:03,435 | ERROR | Operación 15 falló: La reserva no se puede cancelar
2026-11-24 04:48:03,435 | ERROR | Operación 16 falló: [Errno 2] No such file or directory: 'archivo_que_no_existe.txt'


Fin de la demostración: el sistema nunca se detuvo.
